In [0]:
use catalog classic_workspace;
use schema default;

In [0]:
CREATE OR REPLACE TABLE products (
  product_id  BIGINT     COMMENT 'Unique product identifier',
  name        STRING  COMMENT 'Product name',
  category    STRING  COMMENT 'Product category',
  price       DOUBLE  COMMENT 'Unit price in USD'
)
USING DELTA
COMMENT 'Product catalogue — used for schema evolution demos';

In [0]:
INSERT INTO products VALUES
  (1, 'Wireless Headphones', 'Electronics', 79.99),
  (2, 'Standing Desk',       'Furniture',   349.00),
  (3, 'USB-C Hub',           'Electronics', 44.99),
  (4, 'Monitor Stand',       'Furniture',   55.00);

In [0]:
select * from products;

In [0]:
select * from products where category = 'Electronics';

In [0]:
%python
from pyspark.sql import Row

bad_data_1 = [
    Row(product_id=5, name='Mechanical Keyboard',
        category='Electronics', price=129.50, discount_pct=0.10),
    Row(product_id=6, name='Desk Lamp',
        category='Furniture',   price=34.99,  discount_pct=0.05),
]

df_bad_1 = spark.createDataFrame(bad_data_1)

# This write WILL FAIL — discount_pct is not in the table schema
df_bad_1.write \
    .format('delta') \
    .mode('append') \
    .saveAsTable('products')

In [0]:
%python
bad_data_2 = [
    Row(product_id=5, name='Mechanical Keyboard',
        category='Electronics', price='129.50'),   # <-- STRING, not DOUBLE
]

df_bad_2 = spark.createDataFrame(bad_data_2)

# This write WILL FAIL — incompatible type for the price column
df_bad_2.write \
    .format('delta') \
    .mode('append') \
    .saveAsTable('products')

In [0]:
%python
new_products = [
    Row(product_id=5, name='Mechanical Keyboard',
        category='Electronics', price=129.50, discount_pct=0.10),
    Row(product_id=6, name='Desk Lamp',
        category='Furniture',   price=34.99,  discount_pct=0.05),
]

df_new = spark.createDataFrame(new_products)

# mergeSchema allows Delta to add the new column
df_new.write \
    .format('delta') \
    .mode('append') \
    .option('mergeSchema', 'true') \
    .saveAsTable('products')

print('Write with mergeSchema succeeded.')

In [0]:
describe table products;

In [0]:
SELECT product_id, name, price, discount_pct
FROM   products
ORDER  BY product_id;

In [0]:
%python
renamed_products = [
    Row(product_id=7, name='Ergonomic Mouse',
        category='Electronics', price=59.99, discount_rate=0.08),  # renamed column
]

df_renamed = spark.createDataFrame(renamed_products)

df_renamed.write \
    .format('delta') \
    .mode('append') \
    .option('mergeSchema', 'true') \
    .saveAsTable('products')

print('Write succeeded. Now check the schema — you may be surprised.')

In [0]:
describe table products;

In [0]:
ALTER TABLE products
ADD COLUMN supplier STRING COMMENT 'Product supplier name';

In [0]:
describe table products;

In [0]:
describe history products;

In [0]:
ALTER TABLE products
SET TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion'   = '2',
  'delta.minWriterVersion'   = '5'
);

In [0]:
describe table products;

In [0]:
ALTER TABLE products
RENAME COLUMN discount_rate TO discount_rate_v2;

In [0]:
describe table products;

In [0]:
ALTER TABLE products DROP COLUMN supplier;

In [0]:
describe table products;

In [0]:
CREATE OR REPLACE TABLE products_v2 (
  product_id     BIGINT,
  name           STRING,
  price          DOUBLE,
  price_with_tax DOUBLE GENERATED ALWAYS AS (ROUND(price * 1.20, 2))
)
USING DELTA
COMMENT 'Products with a tax-inclusive price computed automatically';

In [0]:
INSERT INTO products_v2 (product_id, name, price) VALUES
  (1, 'Wireless Headphones', 79.99),
  (2, 'Standing Desk',       349.00),
  (3, 'USB-C Hub',           44.99);

In [0]:
describe table products_v2;

In [0]:
SELECT product_id, name, price, price_with_tax
FROM   products_v2
ORDER  BY product_id;